# Phase 1 · Lesson 1 — Pre-training

**Mastering Agentic AI Certification · Pre-read**

> Before an LLM can reason, plan, or act as an *agent*, it must first learn language and world knowledge. That foundational stage is **pre-training**. This lesson summarises what pre-training is, how it works, and why it is the bedrock of every agentic AI system.

## 1. What is pre-training?

**Pre-training** is the first and most compute-intensive stage of building a large language model (LLM). The model is trained on a *massive, broad corpus* of text (and increasingly code, images, and audio) to learn the statistical structure of language and a compressed representation of world knowledge.

It is **self-supervised**: no human labels are required. The training signal comes directly from the data itself — the model repeatedly predicts parts of the text that are hidden from it.

The output of pre-training is a **base (foundation) model** — broadly knowledgeable but not yet aligned to follow instructions or behave as a helpful assistant.

### What does "label" mean here?

In traditional **supervised learning**, every training example is an *(input, label)* pair, where the **label** (a.k.a. the *target* or *ground truth*) is the correct answer a **human has manually annotated**:

| Task | Input | Label (human-provided) |
|------|-------|------------------------|
| Spam detection | an email | `spam` / `not spam` |
| Image classification | a photo | `cat` |
| Sentiment | a review | `positive` |

Producing these labels is slow and expensive, which limits how much data you can use.

**Self-supervised pre-training needs no human labels** because the *answer is already in the raw text* — we just hide part of it and ask the model to predict it:

```
Input:  "The cat sat on the ___"
Target: "mat"      <- this word came straight from the original text
```

So in this context a **label is simply the hidden piece of text the model tries to predict** (the next token, or a masked token). It is derived *automatically from the data itself* rather than written by a human annotator.

That is the whole reason LLMs can scale: **any** piece of text becomes millions of free *(input, hidden-target)* training pairs — no annotation required.

## 2. Where pre-training sits in the LLM lifecycle

```
  [1] PRE-TRAINING            -> base model: learns language + knowledge (self-supervised, huge data)
        |
  [2] FINE-TUNING / SFT       -> instruction-following on curated examples
        |
  [3] ALIGNMENT (RLHF / DPO)  -> helpful, honest, harmless behaviour
        |
  [4] AGENTIC USE             -> tools, memory, planning, multi-step action
```

Pre-training is step **[1]** — everything an agent later does is built on the representations learned here. The later stages *steer* the model; they do not teach it most of its knowledge.

## 3. The core objective: next-token prediction

Most modern LLMs (GPT, Claude, Llama, Gemini) are **autoregressive**: given a sequence of tokens, the model predicts the probability distribution of the *next* token. Training minimises the **cross-entropy loss** between the predicted distribution and the actual next token across trillions of tokens.

$$\mathcal{L} = -\sum_{t} \log P_\theta(x_t \mid x_1, x_2, \dots, x_{t-1})$$

Doing this well at scale forces the model to implicitly learn grammar, facts, reasoning patterns, and even rudimentary planning — all of which the agentic layers later exploit.

## 4. The four ingredients of pre-training

| Ingredient | What it means | Why it matters for agents |
|---|---|---|
| **Data** | Trillions of tokens from web text, books, code, etc. — heavily filtered & deduplicated | Quality/diversity of data sets the ceiling on what an agent can know |
| **Architecture** | The **Transformer** with self-attention | Attention lets the model use long context — essential for tool outputs & memory |
| **Compute** | Thousands of GPUs/TPUs for weeks/months | Determines feasible model & data scale |
| **Scaling laws** | Loss falls predictably as data, params & compute grow | Guides how big a model must be to unlock capabilities |

## 5. Tokenization — how text becomes numbers

Models don't see characters or words; they see **tokens** (sub-word units produced by a tokenizer such as Byte-Pair Encoding). A token is roughly 3–4 characters of English text. Token counts drive **context-window limits** and **API cost** — both critical budget concerns when an agent makes many tool calls in a loop.

The toy code cell below illustrates the *idea* of tokenization and next-token prediction with no API keys or downloads — so it runs anywhere, including this dev container.

In [1]:
# A minimal, offline illustration of next-token prediction.
# This is NOT a real LLM — it's a toy bigram model to make the concept concrete.
from collections import defaultdict, Counter

corpus = (
    "agentic ai plans and acts agentic ai uses tools "
    "agentic ai reasons step by step agentic ai calls tools and acts"
)
tokens = corpus.split()

# Learn P(next_word | current_word) by counting — the same idea as pre-training,
# just with counts instead of a neural network.
transitions = defaultdict(Counter)
for current, nxt in zip(tokens, tokens[1:]):
    transitions[current][nxt] += 1

def predict_next(word):
    counts = transitions[word]
    total = sum(counts.values())
    return {w: round(c / total, 2) for w, c in counts.most_common()}

print("Vocabulary size:", len(set(tokens)))
print("Given 'agentic', next-token distribution:", predict_next("agentic"))
print("Given 'ai',      next-token distribution:", predict_next("ai"))

Vocabulary size: 11
Given 'agentic', next-token distribution: {'ai': 1.0}
Given 'ai',      next-token distribution: {'plans': 0.25, 'uses': 0.25, 'reasons': 0.25, 'calls': 0.25}


In [2]:
# Greedy text generation: repeatedly pick the most likely next token.
# Real LLMs do this too, but with billions of parameters and sampling strategies.
def generate(start, n=8):
    out = [start]
    word = start
    for _ in range(n):
        nxt = transitions[word].most_common(1)
        if not nxt:
            break
        word = nxt[0][0]
        out.append(word)
    return " ".join(out)

print(generate("agentic"))

agentic ai plans and acts agentic ai plans and


## 6. Self-supervision & emergent capabilities

Because pre-training needs no labels, it can consume *internet-scale* data. As models scale up, capabilities **emerge** that were never explicitly trained — in-context learning, chain-of-thought reasoning, translation, and basic tool-use intuition. These emergent abilities are precisely what make **agentic AI** possible: the agent loop (observe → think → act) leans on reasoning that arose during pre-training.

## 7. Why pre-training matters for agentic AI

- **Knowledge & reasoning** an agent uses for planning come from the pre-trained base.
- **Context window** (set by tokenization + architecture) bounds how much tool output, memory, and history an agent can hold.
- **Knowledge cutoff**: pre-training data has a date — agents use **tools / RAG** to fill the gap with fresh information.
- **Cost & latency** of every agent step are driven by token counts established at pre-training.
- You rarely pre-train yourself; you **consume** a foundation model via an API and add fine-tuning, prompting, tools, and orchestration on top.

## 8. Key takeaways

1. **Pre-training** = self-supervised next-token prediction on massive data → a **base/foundation model**.
2. It is the most expensive stage and the source of the model's knowledge and reasoning.
3. Built on the **Transformer**; governed by **scaling laws**.
4. **Tokenization** determines context limits and cost.
5. Fine-tuning, alignment, and the **agentic layer** all build on top of pre-training — they steer, not replace, it.

---
*Next:* fine-tuning & instruction-tuning — how a base model becomes a helpful, controllable assistant.

In [ ]:
# Environment sanity check — confirms the notebook is running in the dev container.
import sys, platform
print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
print("Lesson 1 (Pre-training) notebook is running \u2713")